In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from scipy.special import ellipk, ellipe
from numba import jit
from io import BytesIO

In [ ]:
@jit(nopython = True)
def step(spins, L, J, beta):
  x, y = np.random.randint(0, L, 2)

  neighbor_sum = (spins[(x + 1) % L, y] +
                  spins[(x - 1) % L, y] +
                  spins[x, (y + 1) % L] +
                  spins[x, (y - 1) % L])
  
  delta = 2 * J * neighbor_sum
    
  if (np.random.random() < 1 / (1 + np.exp(-beta * delta))):
    spins[x, y] = 1
  else:
    spins[x, y] = -1


@jit(nopython = True)
def sweep(spins, L, J, beta):
  for _ in range(L**2):
    step(spins, L, J, beta)


def onsager_solution(T, J):
  Tc = 2 * J / np.log(1 + np.sqrt(2))
  magnetization = []
  for temp in T:
    if temp >= Tc:
      magnetization.append(0)
    else:
      m = (1 - (np.sinh(2 * J / temp))**(-4))**(1/8)
      magnetization.append(m)
  return np.array(magnetization)


def heat_solution(T, J):
  heat_cap = []
  for temp in T:
    X = 2 * J / temp
    tanh_X = np.tanh(X)
    kappa_sq = (2 * tanh_X / np.cosh(X))**2
    kappa_prime = 2 * tanh_X**2 - 1

    K = ellipk(kappa_sq)
    E = ellipe(kappa_sq)

    term1 = (1.0 / tanh_X) / temp
    term2 = (2 * K) - (2 * E) - (1 - kappa_prime) * ((np.pi / 2.0) + (kappa_prime * K))

    c = (2.0 / np.pi) * term1 * term2
    heat_cap.append(c)

  return heat_cap


def mag_sus_simulation(spins, J, beta, steps_to_skip, N_steps):
  L = spins.shape[0]
  magnetization = np.zeros(N_steps)
  susceptibility = np.zeros(N_steps)

  for _ in range(steps_to_skip):
    sweep(spins, L, J, beta)

  for i in range(N_steps):
    sweep(spins, L, J, beta)
    magnetization[i] = np.sum(spins)
    susceptibility[i] = np.var(spins) * beta
  
  return np.mean(np.abs(magnetization)) / L**2, np.mean(susceptibility)


def heat_capacity(spins, J, beta, steps_to_skip, N_steps):
  L = spins.shape[0]
  heat = np.zeros(N_steps)

  for _ in range(steps_to_skip):
    sweep(spins, L, J, beta)

  for i in range(N_steps):
    sweep(spins, L, J, beta)

    s_neighbors = (
      np.roll(spins, 1, axis=0) +
      np.roll(spins, -1, axis=0) +
      np.roll(spins, 1, axis=1) +
      np.roll(spins, -1, axis=1)
    )

    energy = -J * spins * s_neighbors
    heat[i] = np.var(energy) * beta**2

  return np.mean(heat)

In [ ]:
J = 1
T = np.linspace(0.01, 5, 50)

In [ ]:
L = 20
beta = 0.27
spins = np.ones((L, L))

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
ax.axis("off")

with imageio.get_writer('../results/gifs/08_ising_model.gif', mode='I', duration=0.1) as writer:
  for _ in range(1000):
    sweep(spins, spins.shape[0], J, beta)

  for _ in range(100):
    ax.clear()

    sweep(spins, spins.shape[0], J, beta)
    ax.imshow(spins, cmap='gray', vmin=-1, vmax=1, interpolation='nearest')
    
    buf = BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    writer.append_data(imageio.imread(buf))

plt.close(fig)  

In [ ]:
for i in [1, 2, 3, 4, 5]:
  print(mag_sus_simulation(np.ones((5, 5)), J, 1 / i, 1000, 5000)[0])

In [ ]:
L10_results = []
L20_results = []
S10_results = []
S20_results = []

for i in T:
  L10, S10 = mag_sus_simulation(np.ones((10, 10)), J, 1 / i, 2000, 5000)
  L20, S20 = mag_sus_simulation(np.ones((20, 20)), J, 1 / i, 2000, 5000)
  
  L10_results.append(L10)
  L20_results.append(L20)
  S10_results.append(S10)
  S20_results.append(S20)

fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(T, L10_results, 'o-', label='L10')
ax.plot(T, L20_results, 's-', label='L20')
ax.plot(T, onsager_solution(T, J), label='onsager')

plt.savefig('../results/plots/08_ising_onsager.png')
ax.set_title("Onsager")
ax.set_xlabel("T")
ax.set_ylabel("Mean magnetization")
ax.legend()
plt.show()
plt.close(fig)


fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(T, S10_results, 'o-', label='L10_sus')
ax.plot(T, S20_results, 's-', label='L20_sus')

plt.savefig('../results/plots/08_ising_susceptibility.png')
ax.set_title("Normed_susceptibility")
ax.set_xlabel("T")
ax.set_ylabel("Susceptibility")
ax.legend()
plt.show()
plt.close(fig)

In [ ]:
H50 = []
for i in T:
  H50.append(heat_capacity(np.ones((10, 10)), J, 1 / i, 2, 50))

fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(T, H50, 'o-', label='H50')
ax.plot(T, heat_solution(T, J), 's-', label='Analitic solution')

plt.savefig('../results/plots/08_ising_heat_capacity_peak.png')
ax.set_title("Heat capacity")
ax.set_xlabel("T")
ax.set_ylabel("Heat capacity")
ax.legend()
plt.show()
plt.close(fig)